# Exploratory Data Analysis (EDA)

Análisis exploratorio de las operaciones procesadas para identificar distribuciones, tendencias y comportamiento de los clientes.

In [0]:
from pyspark.sql import functions as F

In [0]:
DIM_INSTRUMENTO_TABLE = "byma.gold.dim_instrumento"
DIM_FECHA_TABLE = "byma.gold.dim_fecha"
DIM_CLIENTE_TABLE = "byma.gold.dim_cliente"
DIM_CANAL_TABLE = "byma.gold.dim_canal"

In [0]:
fact = spark.table("byma.gold.fact_operaciones")

display(fact.limit(10))

id_transaccion,fecha_sk,canal_sk,instrumento_sk,cliente_sk,fecha,tipoTran,cantidad,precio,monto_total,es_fin_de_semana,flag_outlier_monto
TXNid25owf75935a,20260102,2,1440,35146,2026-01-02T21:55:45.000Z,Compra,3.000000,55137.628400,165412.885200000,false,false
TXNr6k2vmwwyzx4w,20260102,2,49,39048,2026-01-02T13:32:05.000Z,Compra,14.000000,0.668900,9.364600000,false,false
TXNa78rapkianouo,20260102,5,402,38386,2026-01-02T16:02:19.000Z,Compra,24.000000,3225.509100,77412.218400000,false,false
TXN2gqu7dknncptk,20260102,2,304,49204,2026-01-02T20:25:56.000Z,Compra,15.000000,59.841800,897.627000000,false,false
TXNfqla09lzrk79u,20260102,2,1146,26744,2026-01-02T23:35:44.000Z,Compra,24.000000,35.157900,843.789600000,false,false
TXN5d0p92k38p6fv,20260102,2,47,19131,2026-01-02T17:37:15.000Z,Venta,20.000000,1008.444300,20168.886000000,false,false
TXNh3wqa1vrgss8k,20260102,2,868,46750,2026-01-02T21:01:21.000Z,Venta,2.000000,16.217400,32.434800000,false,false
TXN4ns7nvv28ue1m,20260102,2,202,689,2026-01-02T17:25:36.000Z,Compra,1.000000,35162.293300,35162.293300000,false,false
TXNw4vnzvzn2hloi,20260102,2,131,36593,2026-01-02T21:37:13.000Z,Compra,2.000000,9282.977900,18565.955800000,false,false
TXN5aiekx6mj53yx,20260102,2,103,12508,2026-01-02T21:37:08.000Z,Compra,6.000000,4072.465700,24434.794200000,false,false


## Distribución por tipo de operación

In [0]:
display(
    fact.groupBy("tipoTran")
        .count()
        .orderBy(F.desc("count"))
)

tipoTran,count
Compra,70051
Venta,29949


## Top 10 instrumentos

In [0]:
dim_instrumento = spark.table(DIM_INSTRUMENTO_TABLE)

display(
    fact.join(dim_instrumento, "instrumento_sk")
        .groupBy("simbolo_titulo")
        .count()
        .orderBy(F.desc("count"))
        .limit(10)
)

simbolo_titulo,count
AL30,8859
AL30D,4589
MSFT,2336
SPY,2311
AMZN,2054
MELI,1881
PAMP,1668
NVDA,1493
YPFD,1485
GOOGL,1453


## Evolución diaria del monto operado

In [0]:
dim_fecha = spark.table(DIM_FECHA_TABLE)

display(
    fact.join(dim_fecha, "fecha_sk")
        .groupBy("fecha_operacion")
        .agg(F.sum("monto_total").alias("monto_total_dia"))
        .orderBy("fecha_operacion")
)

fecha_operacion,monto_total_dia
2026-01-02,162646407.985400000
2026-01-03,320260603.005600000
2026-01-05,161037987.684300000
2026-01-06,456433333.785800000
2026-01-07,1024693536.581800000
2026-01-08,629883855.566000000
2026-01-09,440937036.134500000
2026-01-10,213125788.944700000
2026-01-12,341913270.216200000
2026-01-13,368940297.428400000


## Clientes más activos

In [0]:
dim_cliente = spark.table(DIM_CLIENTE_TABLE).filter(F.col("is_current") == True)

display(
    fact.join(dim_cliente, "cliente_sk")
        .groupBy("id_cliente")
        .count()
        .orderBy(F.desc("count"))
        .limit(10)
)

id_cliente,count
CLIDBB810DF,676
CLI4F600941,542
CLIBDBA3A65,282
CLID3987826,165
CLI5DB07751,127
CLI7408041E,101
CLIC797296B,87
CLI51E03684,80
CLIA46995EC,77
CLIC1EC8BBC,74
